# Aprire i file convertiti: NPZ, HDF5, OME-TIFF, FITS

Quattro esempi, uno per formato, sullo stesso cubo (mappa11, rebin 512 canali,
calibrato). Il contenuto dei 4 file e' identico: cubo `(y, x, canali)` di
conteggi raw (uint32, lossless) + metadati.

I file sono nella stessa cartella del notebook:

- `mappa11.ome.tif` e `mappa11.fits`: share ownCloud (`512/` e `fits/512/`);
- `mappa11.npz` e `mappa11.h5`: generati dal raw in una riga, se non li hai:

```bash
xrf-raw-to-h5 mappa11.txt -o . --format npz --rebin-to 512 --overwrite
xrf-raw-to-h5 mappa11.txt -o . --format h5  --rebin-to 512 --overwrite
```

Ambiente: `uv pip install "chnet-xrf-raw-data-converter[fits]" h5py matplotlib`
(`[fits]` serve solo per l'esempio FITS; npz e h5 bastano con numpy/h5py).


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

def plot3(img, energy, meta):
    """3 pannelli: mappa totale, media per pixel, spettro sommato."""
    g = meta["grid"]
    ext = [g["x_min"], g["x_max"], g["y_min"], g["y_max"]]
    total = img.sum(axis=-1, dtype=np.int64)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    im = ax[0].imshow(total, extent=ext, origin="lower", cmap="magma")
    ax[0].set_title(f"Total events ({int(total.sum()):,})")
    im2 = ax[1].imshow(img.mean(axis=-1), extent=ext, origin="lower", cmap="viridis")
    ax[1].set_title("Mean per pixel")
    ax[2].plot(energy, img.sum(axis=(0, 1), dtype=np.int64), lw=0.8)
    ax[2].set_title(f"Summed spectrum ({img.shape[-1]} ch)")
    ax[2].set_xlabel("E (keV)"); ax[2].set_ylabel("counts")
    for a in ax[:2]:
        a.set_xlabel("x (µm)"); a.set_ylabel("y (µm)")
    fig.colorbar(im, ax=ax[0]); fig.colorbar(im2, ax=ax[1])
    fig.tight_layout(); plt.show()


## 1) NPZ

`np.load` da numpy: un dizionario di array. `attrs` e' il dizionario di attributi serializzato in JSON (dentro,
`metadata_json` e' il JSON dei metadati completi, stesso schema di tutti gli altri formati);
`energy_axis` esiste solo se calibrato.

In [ ]:
import numpy as np

z = np.load("mappa11.npz")
img = z["img"]                       # (y, x, canali) uint32
energy = z["energy_axis"]            # presente solo se calibrato
attrs = json.loads(z["attrs"].item())          # attrs e' un ndarray 0-dim
meta = json.loads(attrs["metadata_json"])      # metadati completi (grid, ...)
print(img.shape, img.dtype, "eventi:", int(img.sum()))
plot3(img, energy, meta)


## 2) HDF5

`h5py`: dataset `img`, `channel`, `x_axis`, `y_axis`, `mask`, `energy_axis`
(+) attrs sullo stesso schema JSON, incluso `metadata_json` completo.

In [ ]:
import h5py

with h5py.File("mappa11.h5", "r") as f:
    print("dataset:", list(f.keys()))
    img = f["img"][...]              # (y, x, canali) uint32
    energy = f["energy_axis"][...]   # presente solo se calibrato
    meta = json.loads(f.attrs["metadata_json"])
print(img.shape, img.dtype, "eventi:", int(img.sum()))
plot3(img, energy, meta)


## 3) OME-TIFF

`read_ome` di xrfraw restituisce un `Dataset` pronto: `.img`, `.energy`
(= m_rebinned·j + q_rebinned), `.metadata`. La calibrazione vive negli
attributi OME (`CalibrationM`/`CalibrationQ`/`Calibrated`).

In [ ]:
from xrfraw import read_ome

ds = read_ome("mappa11.ome.tif")
print(ds.img.shape, ds.img.dtype, "eventi:", int(ds.img.sum()))
print("calibrato:", ds.calibrated, "m:", ds.calibration_m, "q:", ds.calibration_q)
plot3(ds.img, ds.energy, ds.metadata)


## 4) FITS

`read_fits` di xrfraw restituisce lo stesso `Dataset`. La calibrazione e'
WCS sull'asse energia (`CTYPE3=ENERGY`, keV) + extension `XRF_META` (JSON) e
`ENERGY` (asse). Richiede l'extra `[fits]` (astropy).

In [ ]:
from xrfraw import read_fits

ds = read_fits("mappa11.fits")
print(ds.img.shape, ds.img.dtype, "eventi:", int(ds.img.sum()))
print("calibrato:", ds.calibrated, "m:", ds.calibration_m, "q:", ds.calibration_q)
plot3(ds.img, ds.energy, ds.metadata)


## Verifica

Per tutti e 4 i formati il totale eventi deve coincidere con il raw
(mappa11: **19.825.090**). I file non calibrati non hanno `energy_axis`/
`ds.energy`: in quel caso l'asse 3 sono i canali e la conversione in energia
si fa a mano con la m/q nota.

Schema completo dei metadati: `output_schema.md`.
